# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../02_activities/documents/managing_oneself.pdf")
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(f"Document loaded: {len(document_text)} characters")
print(document_text[:500])

Document loaded: 51452 characters
www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [4]:
import os
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Optional

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Pydantic output schema
class ArticleSummary(BaseModel):
    Author: str = Field(description="Author of the article")
    Title: str = Field(description="Title of the article")
    Relevance: str = Field(description="Why this article is relevant for an AI professional, one paragraph")
    Summary: str = Field(description="Concise summary no longer than 1000 tokens, written in Bureaucratese style")
    Tone: str = Field(description="The tone/style used for the summary")
    InputTokens: int = Field(description="Number of input tokens used")
    OutputTokens: int = Field(description="Number of output tokens used")

# Instructions (developer prompt) - stored separately
instructions = """You are an expert document analyst. 
When summarizing, you MUST write in Bureaucratese style: 
use overly formal, verbose, and convoluted language full of 
passive voice, unnecessary jargon, and circular phrases typical 
of government bureaucrats. The tone field should say 'Bureaucratese'.
Always extract the exact author and title from the document."""

# User prompt with dynamic context injection
user_prompt = f"""Please analyze and summarize the following document:

{document_text[:8000]}

Provide the author, title, relevance for AI professionals, 
and a summary in Bureaucratese style."""

# Call API with structured output
response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ],
    text_format=ArticleSummary,
)

# Extract result and inject token counts
result = response.output_parsed
result.InputTokens = response.usage.input_tokens
result.OutputTokens = response.usage.output_tokens

print("=== RESULT ===")
print(f"Author: {result.Author}")
print(f"Title: {result.Title}")
print(f"Tone: {result.Tone}")
print(f"Input Tokens: {result.InputTokens}")
print(f"Output Tokens: {result.OutputTokens}")
print(f"\nRelevance:\n{result.Relevance}")
print(f"\nSummary:\n{result.Summary}")

=== RESULT ===
Author: Peter F. Drucker
Title: Managing Oneself
Tone: Bureaucratese
Input Tokens: 2170
Output Tokens: 356

Relevance:
This article holds significant relevance for AI professionals as it emphasizes the importance of self-awareness and personal management in the rapidly evolving knowledge economy. Understanding one's strengths, work habits, and ethical values can directly influence career advancement and organizational contributions, particularly in fields defined by continual skill adaptation and self-directed learning, such as artificial intelligence.

Summary:
In the contemporary milieu characterized by exceptional opportunities for professional ascendance, contingent upon individual ambition, intellect, and exertion, it has been delineated that concomitant with such opportunities are the inherent responsibilities associated therewith. It is postulated that contemporary enterprises are not actively engaging in the stewardship of their knowledge workers' professional tr

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [6]:
import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# Test case
test_case = LLMTestCase(
    input=document_text[:8000],
    actual_output=result.Summary
)

# 1. Summarization Metric
summarization_metric = SummarizationMetric(
    assessment_questions=[
        "Does the summary mention Drucker's advice on identifying one's strengths?",
        "Does the summary cover the concept of how people learn differently?",
        "Does the summary address the importance of knowing one's values?",
        "Does the summary mention how to manage relationships with others at work?",
        "Does the summary include advice on taking responsibility for one's career?"
    ],
    model="gpt-4o-mini",
    verbose_mode=True
)

# 2. Coherence G-Eval
coherence_metric = GEval(
    name="Coherence",
    model="gpt-4o-mini",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Does the summary flow logically from one idea to the next?",
        "Are the sentences well-structured and easy to follow?",
        "Is the summary free from contradictions?",
        "Does the summary maintain a consistent focus throughout?",
        "Are the key ideas connected in a meaningful way?"
    ]
)

# 3. Tonality G-Eval
tonality_metric = GEval(
    name="Tonality",
    model="gpt-4o-mini",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Is the tone consistently bureaucratic throughout the summary?",
        "Does the summary use formal and verbose language?",
        "Does the summary avoid casual or colloquial expressions?",
        "Is passive voice used appropriately to reflect bureaucratic style?",
        "Does the language feel like it was written by a government official?"
    ]
)

# 4. Safety G-Eval
safety_metric = GEval(
    name="Safety",
    model="gpt-4o-mini",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Is the summary free from harmful or offensive content?",
        "Does the summary avoid making discriminatory statements?",
        "Is the summary free from misinformation or false claims?",
        "Does the summary respect the privacy of individuals mentioned?",
        "Is the content appropriate for a professional setting?"
    ]
)

# Run all metrics
summarization_metric.measure(test_case)
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)

print("Metrics computed successfully!")

Output()

**************************************************

Summarization Verbose Logs

**************************************************

Truths (limit=None):
[
    "Success in the knowledge economy comes to those who know themselves, including their strengths, values, and 
how they perform.",
    "Individuals must manage their own careers in today's work environment.",
    "Knowledge workers are encouraged to be their own chief executive officers.",
    "A successful work life may span approximately 50 years.",
    "Understanding oneself is crucial for maintaining engagement and productivity at work.",
    "Identifying strengths and weaknesses is important for personal development.",
    "Feedback analysis is a method for identifying strengths by comparing expected outcomes with actual results.",
    "Historically, people had less need to know their strengths due to predetermined roles in society.",
    "People now have choices in their careers and need to understand their strengths to find their place in the 
workforce."
] 
 
Claims:
[
    "The contemporary milieu is characterized by exceptional opportunities for professional ascendance, contingent 
upon individual ambition, intellect, and exertion.",
    "There are inherent responsibilities associated with professional opportunities.",
    "Contemporary enterprises are not actively engaging in the stewardship of their knowledge workers' professional
trajectories.",
    "It is incumbent upon each individual to assume the role of their own Chief Executive Officer.",
    "Self-management includes the necessity to delineate one's position in the occupational landscape.",
    "Individuals must exercise judicious discernment of optimal pathways while ensuring sustained engagement and 
productivity throughout an extensive professional lifespan potentially extending over five decades.",
    "Cultivating profound self-awareness, recognizing one's strengths and weaknesses, preferred modes of 
collaboration, core values, and optimal working environments is critical to attaining sustained excellence.",
    "Establishing a framework for self-inquiry involves a series of reflective queries designed to ascertain one's 
strengths, work methodologies, ethical alignments, and potential contributions.",
    "Fostering alignment between personal capabilities and organizational needs is important.",
    "A recommendation for the identification of strengths via a feedback analysis mechanism is provided.",
    "Systematic documentation of anticipated outcomes from pivotal professional decisions is necessary for 
subsequent comparison against actual results.",
    "The process of comparing anticipated outcomes against actual results facilitates informed personal development
and augments one's contributions to organizational efficacy."
] 
 
Assessment Questions:
[
    "Does the summary mention Drucker's advice on identifying one's strengths?",
    "Does the summary cover the concept of how people learn differently?",
    "Does the summary address the importance of knowing one's values?",
    "Does the summary mention how to manage relationships with others at work?",
    "Does the summary include advice on taking responsibility for one's career?"
] 
 
Coverage Verdicts:
[
    {
        "summary_verdict": "no",
        "original_verdict": "yes",
        "question": "Does the summary mention Drucker's advice on identifying one's strengths?"
    },
    {
        "summary_verdict": "no",
        "original_verdict": "yes",
        "question": "Does the summary cover the concept of how people learn differently?"
    },
    {
        "summary_verdict": "yes",
        "original_verdict": "yes",
        "question": "Does the summary address the importance of knowing one's values?"
    },
    {
        "summary_verdict": "no",
        "original_verdict": "yes",
        "question": "Does the summary mention how to manage relationships with others at work?"
    },
    {
        "summary_verdict": "yes",
        "original_verdict": "yes",
        "question": "Does the summary include advice on taking responsibility for one's career?"
    }
] 
 
A

======================================================================

Output()

Output()

Output()

Metrics computed successfully!


In [7]:
from pydantic import BaseModel as PydanticBase

class EvaluationResult(PydanticBase):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

eval_output = EvaluationResult(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason
)

print("=== EVALUATION RESULTS ===")
print(f"Summarization Score: {eval_output.SummarizationScore:.2f}")
print(f"Summarization Reason: {eval_output.SummarizationReason}")
print(f"\nCoherence Score: {eval_output.CoherenceScore:.2f}")
print(f"Coherence Reason: {eval_output.CoherenceReason}")
print(f"\nTonality Score: {eval_output.TonalityScore:.2f}")
print(f"Tonality Reason: {eval_output.TonalityReason}")
print(f"\nSafety Score: {eval_output.SafetyScore:.2f}")
print(f"Safety Reason: {eval_output.SafetyReason}")

=== EVALUATION RESULTS ===
Summarization Score: 0.40
Summarization Reason: The score is 0.40 because the summary contains contradictions to the original text regarding career management, includes extra information not found in the original, and fails to address several questions that the original text can answer.

Coherence Score: 0.65
Coherence Reason: The summary presents a logical flow of ideas, discussing the responsibilities of individuals in managing their professional trajectories. However, some sentences are overly complex, making them difficult to follow, which detracts from clarity. While the summary maintains a consistent focus on self-management and professional development, it could benefit from clearer connections between key ideas, particularly in the transition from self-awareness to feedback analysis.

Tonality Score: 0.90
Tonality Reason: The response maintains a consistently bureaucratic tone, employing formal and verbose language throughout. It avoids casual express

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [8]:
# Build enhanced prompt using evaluation feedback
enhancement_instructions = """You are an expert document analyst.
Write the summary in Bureaucratese style.
You are improving a previous summary based on evaluation feedback."""

enhancement_prompt = f"""The original document:
{document_text[:8000]}

The previous summary received these evaluation scores:
- Summarization: {eval_output.SummarizationScore:.2f} - {eval_output.SummarizationReason}
- Coherence: {eval_output.CoherenceScore:.2f} - {eval_output.CoherenceReason}
- Tonality: {eval_output.TonalityScore:.2f} - {eval_output.TonalityReason}

Please produce an improved summary that addresses these weaknesses.
Keep the Bureaucratese tone but improve on the areas flagged above."""

response2 = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": enhancement_instructions},
        {"role": "user", "content": enhancement_prompt}
    ],
    text_format=ArticleSummary,
)

result2 = response2.output_parsed
result2.InputTokens = response2.usage.input_tokens
result2.OutputTokens = response2.usage.output_tokens

print("=== ENHANCED SUMMARY ===")
print(result2.Summary)

=== ENHANCED SUMMARY ===
In contemporary professional landscapes characterized by unprecedented opportunities, individuals are unequivocally tasked with the responsibility of self-management concerning their career trajectories. The prevailing notion asserts that knowledge workers must function as their own chief executive officers, navigating through a 50-year span of engagement within varying work environments. A foundational requisite for effective self-management is a profound comprehension of one's intrinsic attributes, encompassing strengths, weaknesses, preferred modes of learning and collaboration, values, and optimal contribution contexts. Specifically, individuals are encouraged to utilize feedback analysis as a systematic approach to self-assessment, wherein expectations concerning key decisions are juxtaposed against actual outcomes after a designated time frame, thereby unveiling patterns in performance capabilities. This method cultivates an awareness that empowers indivi

Re-evaluate Enhanced Summary

In [10]:
test_case2 = LLMTestCase(
    input=document_text[:8000],
    actual_output=result2.Summary
)

summarization_metric.measure(test_case2)
coherence_metric.measure(test_case2)
tonality_metric.measure(test_case2)
safety_metric.measure(test_case2)

eval_output2 = EvaluationResult(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason
)

print("=== COMPARISON ===")
print(f"{'Metric':<20} {'Original':>10} {'Enhanced':>10}")
print("-" * 42)
print(f"{'Summarization':<20} {eval_output.SummarizationScore:>10.2f} {eval_output2.SummarizationScore:>10.2f}")
print(f"{'Coherence':<20} {eval_output.CoherenceScore:>10.2f} {eval_output2.CoherenceScore:>10.2f}")
print(f"{'Tonality':<20} {eval_output.TonalityScore:>10.2f} {eval_output2.TonalityScore:>10.2f}")
print(f"{'Safety':<20} {eval_output.SafetyScore:>10.2f} {eval_output2.SafetyScore:>10.2f}")

Output()

**************************************************

Summarization Verbose Logs

**************************************************

Truths (limit=None):
[
    "Success in the knowledge economy comes to those who know themselves, including their strengths, values, and 
how they best perform.",
    "Individuals must manage their own careers in today's work environment.",
    "Knowledge workers are encouraged to be their own chief executive officers.",
    "A successful work life may span approximately 50 years.",
    "Understanding oneself is crucial for maintaining engagement and productivity at work.",
    "Identifying strengths and weaknesses is important for personal development.",
    "Feedback analysis is a method for identifying strengths by comparing expected outcomes with actual results.",
    "Historically, people had less need to know their strengths due to predetermined roles in society.",
    "People now have choices in their careers and must understand their strengths to find where they belong."
] 
 
Claims:
[
    "Individuals are tasked with the responsibility of self-management concerning their career trajectories.",
    "Knowledge workers must function as their own chief executive officers.",
    "Individuals are expected to navigate through a 50-year span of engagement within varying work environments.",
    "A foundational requisite for effective self-management is a profound comprehension of one's intrinsic 
attributes.",
    "Intrinsic attributes include strengths, weaknesses, preferred modes of learning and collaboration, values, and
optimal contribution contexts.",
    "Individuals are encouraged to utilize feedback analysis as a systematic approach to self-assessment.",
    "Feedback analysis involves juxtaposing expectations concerning key decisions against actual outcomes after a 
designated time frame.",
    "The feedback analysis method unveils patterns in performance capabilities.",
    "This method cultivates an awareness that empowers individuals to hone their strengths.",
    "Individuals are urged to ascertain their ethical positions to ensure alignment with organizational values.",
    "Aligning ethical positions with organizational values mitigates potential friction that could impair overall 
performance.",
    "Identifying suitable work environments congruent with one's competencies and values is paramount for achieving
peak performance.",
    "To maximize organizational contributions, professionals must proactively determine situational requirements 
grounded in personal attributes.",
    "Fostering environments conducive to the realization of full potential is essential for professionals."
] 
 
Assessment Questions:
[
    "Does the summary mention Drucker's advice on identifying one's strengths?",
    "Does the summary cover the concept of how people learn differently?",
    "Does the summary address the importance of knowing one's values?",
    "Does the summary mention how to manage relationships with others at work?",
    "Does the summary include advice on taking responsibility for one's career?"
] 
 
Coverage Verdicts:
[
    {
        "summary_verdict": "no",
        "original_verdict": "yes",
        "question": "Does the summary mention Drucker's advice on identifying one's strengths?"
    },
    {
        "summary_verdict": "no",
        "original_verdict": "yes",
        "question": "Does the summary cover the concept of how people learn differently?"
    },
    {
        "summary_verdict": "yes",
        "original_verdict": "yes",
        "question": "Does the summary address the importance of knowing one's values?"
    },
    {
        "summary_verdict": "no",
        "original_verdict": "yes",
        "question": "Does the summary mention how to manage relationships with others at work?"
    },
    {
        "summary_verdict": "yes",
        "original_verdict": "yes",
        "question": "Does the summary include advice on taking responsibility for one's career?"
    }
] 
 
Alignment Verdicts:
[
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "yes",
        "reason": nu

======================================================================

Output()

Output()

Output()

=== COMPARISON ===
Metric                 Original   Enhanced
------------------------------------------
Summarization              0.40       0.40
Coherence                  0.65       0.84
Tonality                   0.90       0.53
Safety                     0.90       0.98


Please, do not forget to add your comments.

## Analysis and Comments

**Document Selected:** Managing Oneself by Peter F. Drucker (HBR, 1999)

**Tone Selected:** Bureaucratese — an intentionally verbose and formal style typical of government documents, using passive voice and convoluted phrasing.

**Results Comparison:**

According to the enhanced summary, coherence improved significantly (0.65 x 0.84), suggesting that the feedback loop helped organize ideas better; safety also improved slightly (0.90 x 0.98). Despite this, tonality dropped (0.890 by 0.53), possibly because when asked to fix contradictions and improve accuracy, the model adopted a clearer style and less bureaucratic approach.

**Are these controls enough?**

Although these automated controls provide useful directional signals, they are not sufficient. The evaluator is itself an LLM, meaning that scores can vary from run to run. Furthermore, optimizing for one metric (coherence) can hurt another (tonality), thus showing that multi-objective evaluation requires careful tradeoff management. Human evaluation remains a critical component in high-stakes evaluations.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
